In [0]:
# ------------------------------------------------------------------------------------------------------------------------------------------
# SCRIPT: 2.4_densidade_eventos_semanal
# LOCAL: 3_gold/src/2_calendario_eventos/
# OBJETIVO: Densidade de eventos e semanas sem atividade a partir da tabela gerada pelo script 3_gold/2_calendario_eventos/2.1_fato_eventos
# ENTRAGA: Densidade de eventos por semana e identificação de semanas sem atividade
# ------------------------------------------------------------------------------------------------------------------------------------------

from pyspark.sql.functions import col, count, weekofyear

# Carregar a tabela fato com a dim_data integrada
df_fato = spark.table("gold_fato_eventos")

df_densidade = df_fato.withColumn("semana_do_ano", weekofyear(col("id_data"))) \
    .groupBy("semana_do_ano") \
    .agg(count("id_evento").alias("quantidade_eventos")) \
    .orderBy("semana_do_ano")


# Criar base com TODAS as semanas do período
df_todas_semanas = spark.table("gold_dim_data") \
    .filter("id_data >= '2025-01-01' AND id_data <= '2025-05-31'") \
    .withColumn("semana_do_ano", weekofyear(col("id_data"))) \
    .select("semana_do_ano").distinct()

# Left Join para manter as semanas com zero eventos
df_densidade_final = df_todas_semanas.join(df_densidade, "semana_do_ano", "left") \
    .fillna(0, subset=["quantidade_eventos"]) \
    .orderBy("semana_do_ano")

# Salvar na Gold
df_densidade_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_densidade_eventos_semanal")

# Exibir o resultado
display(df_densidade_final)
 